In [ ]:
import numpy as np
from scipy.constants import R
from scipy.special import logsumexp
import warnings

import matplotlib.pyplot as plt

from bussilab.wham import wham
from mg_sampling.tools import read_plumed

path = '../replex/7ez0/optimize/12_replicas/mMg_tip3p'

RT = R * 300 * 1e-3
scalings = np.loadtxt(f'{path}/scalings.txt')
nrep = len(scalings)
start = 500

U_kN = [
    read_plumed(f'{path}/rep{k:02d}/bias.COLVAR.{k}')['cv.tot'][start:]
    for k in range(nrep)
]
U_n = np.concatenate(U_kN)
U_nk = scalings * U_n[:, None]
W = wham(U_nk, T=RT)

def w_n(λ):
    logw_n = W.logW - λ * U_n / RT
    logw_n -= logsumexp(logw_n)
    return np.exp(logw_n)

def w_nk(λ_k):
    logw_nk = W.logW[:, None] - λ_k * U_n[:, None] / RT
    logw_nk -= logsumexp(logw_nk, axis=0)
    return np.exp(logw_nk)

In [ ]:
def p_acc(w_nk, i, j, stride=1):
    w_i = w_nk[::stride, i]
    w_j = w_nk[::stride, j]
    w_i /= w_i.sum()
    w_j /= w_j.sum()
    ww = w_i[:, None] * w_j[None, :]
    return np.sum(np.minimum(ww, ww.T))

def read_p_acc(path):
    with open(path, 'r') as f:
        for line in f:
            if line.startswith("Repl  average probabilities:"):
                next(f)
                p_acc = np.array(next(f).split()[1:], dtype=float)
                return p_acc
    raise ValueError("Average probabilities not found in log file.")

In [ ]:
empirical_p_acc = read_p_acc(f'{path}/rep00/run.log')
plt.plot([p_acc(w_nk(scalings), k, k+1, stride=10) for k in range(11)])
plt.plot(empirical_p_acc)

In [ ]:
def loss(scalings):
    _scalings = np.concatenate([[0], scalings, [1]])
    _w_nk = w_nk(_scalings)
    _p_acc = [p_acc(_w_nk, k, k+1, stride=10) for k in range(11)]
    return -np.prod(_p_acc)

In [ ]:
from scipy.optimize import minimize

In [ ]:
bounds = 10 * [(0, 1)]
res = minimize(loss, scalings[1:-1], bounds=bounds, method='Nelder-Mead')
opt_scalings = np.concatenate([[0], res.x, [1]])

In [ ]:
empirical_p_acc = read_p_acc(f'{path}/rep00/run.log')
plt.plot([p_acc(w_nk(scalings), k, k+1, stride=1) for k in range(11)])
plt.plot([p_acc(w_nk(opt_scalings), k, k+1, stride=1) for k in range(11)])
plt.plot(empirical_p_acc)

In [ ]:
plt.plot(scalings)
plt.plot(opt_scalings)